# 🔢 Notebook 06: Sequence Creation & Train/Test Split

**Objective:** Create sliding window sequences and split into train/test sets.

- Window size: 24 hours (lookback)
- Target: AQI category code at next timestep
- Split: 70% train / 30% test (chronological)

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config, load_csv
from src.preprocessing.scaler import AQIScaler
from src.preprocessing.sequence_generator import (
    create_sequences, split_sequences, save_sequences
)

config = load_config('../config/config.yaml')
df = load_csv('../' + config['data']['featured_path'])
print(f"Loaded featured data: {df.shape}")

## 6.1 Identify Feature and Target Columns

In [ ]:
# Target column
target_col = 'aqi_code'

# Exclude non-feature columns
exclude_cols = ['aqi', 'aqi_category', 'aqi_code']
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Feature columns: {len(feature_cols)}")
print(f"Target column: {target_col}")
print(f"Target distribution:")
print(df[target_col].value_counts().sort_index())

## 6.2 Z-Score Normalization

In [ ]:
# Split BEFORE scaling to avoid data leakage
split_idx = int(len(df) * 0.70)
df_train = df.iloc[:split_idx]
df_test = df.iloc[split_idx:]

print(f"Train: {len(df_train)} rows")
print(f"Test:  {len(df_test)} rows")

# Fit scaler on training data only
scaler = AQIScaler()
scaler.fit(df_train, feature_cols)

# Transform both
df_train_scaled = scaler.transform(df_train)
df_test_scaled = scaler.transform(df_test)

# Save scaler
os.makedirs('../artifacts', exist_ok=True)
scaler.save('../artifacts/scaler.joblib')

print(f"\nScaling stats (first 5 features):")
print(scaler.get_stats().head())

## 6.3 Create Sequences

In [ ]:
window_size = config['sequence']['window_size']  # 24

# Create train sequences
X_train, y_train = create_sequences(
    df_train_scaled[feature_cols].values,
    df_train_scaled[target_col].values.astype(int),
    window_size=window_size
)

# Create test sequences
X_test, y_test = create_sequences(
    df_test_scaled[feature_cols].values,
    df_test_scaled[target_col].values.astype(int),
    window_size=window_size
)

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_test:  {y_test.shape}")

In [ ]:
# Save sequences
save_sequences(X_train, X_test, y_train, y_test,
               '../' + config['data']['sequences_path'])

print("\nSequences saved successfully!")

## ✅ Sequence Creation Complete

**Next:** Notebook 07 — CNN Model Training